# 01 — TF-IDF + LogisticRegression 기준선

> **선택 실행용 실험 기반 코드입니다.** 최신 최종 실험 명세 전체를 구현한 상태는 아닙니다. 실제 본 실험 전 [`docs/ml/README.md`의 구현 차이](../../../docs/ml/README.md#implementation-gaps)를 확인하고 필요한 보완을 완료하세요. 이 노트북의 설정을 최종 연구 기준으로 사용하지 않습니다.

봉인된 train/validation만 읽어 T1 주 기준선과 T2 민감도 기준선을 만듭니다. test와 RouterArena는 이 노트북에서 읽지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
archive = '/content/drive/MyDrive/LLM_Router/colab-inputs/router-model-comparison-colab-code.tar.gz'

In [ ]:
!rm -rf /content/LLM_Router
!tar -xzf "{archive}" -C /content
%cd /content/LLM_Router
!pip -q install -r ml/requirements-comparison.lock
!pip -q install --no-deps -e ml
import json, os, pathlib, shutil
print('TF-IDF 환경 준비 완료')

In [ ]:
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/LLM_Router/router-model-comparison/router-model-comparison-v1')
DATASET = DRIVE_ROOT / 'dataset'
if not (DATASET / '_SUCCESS').is_file():
    raise RuntimeError('00 노트북의 봉인된 dataset/_SUCCESS가 없습니다.')
manifest = json.loads((DATASET / 'manifest.json').read_text())
resolution = json.loads((DATASET / 'coverage/resolution.json').read_text())
if manifest['coverage_resolution_status'] != 'complete' or resolution['status'] != 'complete':
    raise RuntimeError('coverage resolution이 완료되지 않았습니다.')
print('dataset:', manifest['dataset_version'])
print('T1/T2:', manifest['selected_thresholds'])

In [ ]:
# Drive I/O를 줄이되 test 파일은 복사하지 않습니다.
LOCAL_DATASET = pathlib.Path('/content/router-model-comparison-v1/dataset')
if LOCAL_DATASET.exists():
    shutil.rmtree(LOCAL_DATASET)
for relative in [
    'manifest.json', 'experiment-config.json', 'coverage/resolution.json',
    'train/training.parquet', 'validation/validation.parquet',
    'train/training-t2.parquet', 'validation/validation-t2.parquet',
]:
    source = DATASET / relative
    target = LOCAL_DATASET / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
print('train/validation 전용 입력 복사 완료')

In [ ]:
LOCAL_RUNS = pathlib.Path('/content/router-model-comparison-v1/runs/tfidf-logreg')
!routing-tfidf-baseline --training-path {LOCAL_DATASET}/train/training.parquet --validation-path {LOCAL_DATASET}/validation/validation.parquet --dataset-manifest-path {LOCAL_DATASET}/manifest.json --experiment-config-path {LOCAL_DATASET}/experiment-config.json --output-dir {LOCAL_RUNS}/t1-seed-42 --threshold-role t1
!routing-tfidf-baseline --training-path {LOCAL_DATASET}/train/training-t2.parquet --validation-path {LOCAL_DATASET}/validation/validation-t2.parquet --dataset-manifest-path {LOCAL_DATASET}/manifest.json --experiment-config-path {LOCAL_DATASET}/experiment-config.json --output-dir {LOCAL_RUNS}/t2-seed-42 --threshold-role t2
for role in ('t1', 't2'):
    metrics = json.loads((LOCAL_RUNS / f'{role}-seed-42/metrics.json').read_text())
    print(role.upper(), json.dumps(metrics['overall'], indent=2, ensure_ascii=False))

In [ ]:
# 각 run을 staging에서 완성한 뒤 한 번에 게시합니다.
DRIVE_RUNS = DRIVE_ROOT / 'runs/tfidf-logreg'
for role in ('t1', 't2'):
    source = LOCAL_RUNS / f'{role}-seed-42'
    staging = DRIVE_RUNS / f'{role}-seed-42.staging'
    final = DRIVE_RUNS / f'{role}-seed-42'
    if staging.exists():
        shutil.rmtree(staging)
    staging.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, staging)
    if final.exists():
        old = json.loads((final / 'manifest.json').read_text())
        new = json.loads((staging / 'manifest.json').read_text())
        if old['artifacts'] != new['artifacts']:
            raise RuntimeError(f'기존 {role} run과 checksum이 다릅니다.')
        shutil.rmtree(staging)
    else:
        os.replace(staging, final)
    print('게시 완료:', final)

## 다음 작업

T1/T2 기준선 결과를 보존하세요. BGE + kNN, ModernBERT 표현 비교와 봉인 평가의 후속 구현은 [ML 최종 실험 명세](../../../docs/ml/README.md#implementation-gaps)를 따릅니다.
